# Phase 6: Execution, Market Microstructure & Systems  
## Notebook 06_05 — Rough Volatility Estimation

### Research question

How can we estimate whether volatility is “rough” from realised volatility data, and why does roughness matter for forecasting, risk, execution, and option pricing?

This notebook follows:

```text
06_01_almgren_chriss_optimal_execution
06_02_dynamic_programming_execution
06_03_square_root_market_impact_model
06_04_hawkes_process_order_flow
06_05_rough_volatility_estimation
```

The previous notebooks focused on execution and order-flow microstructure. This notebook studies volatility from a microstructure-adjacent perspective:

> realised volatility is jagged, persistent, clustered, and often empirically rough.

Rough volatility models are motivated by the empirical observation that log-volatility paths appear less regular than Brownian motion, with Hurst exponents often estimated well below $0.5$.

It covers:

1. smooth versus rough volatility;
2. Hurst exponent intuition;
3. fractional Gaussian noise;
4. rough log-volatility simulation;
5. smooth volatility benchmark;
6. intraday return simulation;
7. realised variance and realised volatility;
8. log-realised-volatility variogram;
9. Hurst estimation by scaling law;
10. bootstrap confidence intervals;
11. estimator sensitivity to lag range;
12. rough versus smooth diagnostics;
13. microstructure noise bias;
14. pre-averaging intuition;
15. volatility forecasting implications;
16. HAR-RV comparison;
17. rough-volatility-inspired forecast;
18. execution-risk implications;
19. volatility-of-volatility diagnostics;
20. governance flags;
21. saved outputs and manifest.

The key idea:

> Rough volatility estimation asks how the increments of log-volatility scale with time; if they scale like $\Delta^H$ with $H<0.5$, volatility is rougher than Brownian motion.

## 1. Smooth versus rough volatility

Classical stochastic-volatility models often imply volatility paths that are fairly smooth or Brownian-like.

Rough volatility says that log-volatility behaves more like a fractional process:

$$
\log \sigma_t \sim W_t^H
$$

where $W_t^H$ is fractional Brownian motion with Hurst exponent $H$.

Interpretation:

| Hurst $H$ | Behaviour |
|---:|---|
| $H=0.5$ | Brownian-like roughness |
| $H<0.5$ | rougher, more jagged, anti-persistent increments |
| $H>0.5$ | smoother, persistent increments |

Empirical rough-volatility literature often finds $H$ around $0.1$ for log-volatility proxies, although estimates are sensitive to data, sampling, noise, and methodology.

## 2. Variogram scaling

For log-volatility $X_t = \log RV_t$, define the empirical variogram:

$$
m(q, \Delta) = E\Big[ |X_{t+\Delta}-X_t|^q \Big]
$$

For a rough volatility process:

$$
m(q,\Delta) \propto \Delta^{qH}
$$

Taking logs:

$$
\begin{aligned}
\log m(q,\Delta) &= c \\
&\quad + qH \log \Delta
\end{aligned}
$$

So for $q=2$:

$$
slope \approx 2H
$$

and:

$$
H \approx \frac{slope}{2}
$$

This notebook estimates $H$ from log-realised-volatility variograms.

## 3. Why roughness matters

Rough volatility matters because it affects:

### Forecasting

Short-horizon volatility dynamics can be more jagged than smooth models assume.

### Options

Rough volatility helps explain steep short-maturity implied-volatility skews.

### Risk

Volatility shocks can cluster and propagate across horizons.

### Execution

Execution cost and impact models depend on volatility. If volatility is rough, intraday execution risk can change quickly.

### Model governance

A volatility model that assumes smooth dynamics may underreact to fast volatility changes.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class RoughVolatilityConfig:
    n_days: int = 1_400
    intraday_steps: int = 78
    seed: int = 42
    annualisation: int = 252
    true_rough_hurst: float = 0.12
    smooth_hurst_proxy: float = 0.50
    base_ann_vol: float = 0.20
    vol_of_logvol: float = 0.75
    drift_ann: float = 0.04
    microstructure_noise_bps: float = 1.0
    max_variogram_lag: int = 60
    bootstrap_samples: int = 500
    forecast_train_frac: float = 0.70
    output_subdir: str = "rough_volatility_estimation_v1"

config = RoughVolatilityConfig()
config

## 5. Fractional Gaussian noise covariance

Fractional Gaussian noise is the increment process of fractional Brownian motion.

For unit-spaced increments:

$$
\begin{aligned}
\gamma(k) &= \frac{1}{2} \Big( |k+1|^{2H} \\
&\quad - 2|k|^{2H} \\
&\quad + |k-1|^{2H} \Big)
\end{aligned}
$$

We use this covariance to simulate a fractional noise sequence.

For clarity and portability, this notebook uses Cholesky simulation. It is fine for educational scale, but production simulation should use faster methods such as Davies-Harte.

In [ ]:
def fgn_covariance(n, hurst):
    k = np.arange(n)
    gamma = 0.5 * (
        np.abs(k + 1) ** (2 * hurst)
        - 2 * np.abs(k) ** (2 * hurst)
        + np.abs(k - 1) ** (2 * hurst)
    )

    cov = np.empty((n, n))
    for i in range(n):
        cov[i, :] = gamma[np.abs(np.arange(n) - i)]
    return cov

def simulate_fractional_gaussian_noise(n, hurst, rng):
    cov = fgn_covariance(n, hurst)
    jitter = 1e-10 * np.eye(n)
    L = np.linalg.cholesky(cov + jitter)
    z = rng.standard_normal(n)
    return L @ z

def simulate_fractional_brownian_motion(n, hurst, rng):
    fgn = simulate_fractional_gaussian_noise(n, hurst, rng)
    fbm = np.cumsum(fgn)
    fbm = (fbm - fbm.mean()) / fbm.std(ddof=1)
    return fbm, fgn

# Quick diagnostic.
rng = np.random.default_rng(config.seed)
fbm_test, fgn_test = simulate_fractional_brownian_motion(300, config.true_rough_hurst, rng)

pd.Series({
    "fbm_mean": fbm_test.mean(),
    "fbm_std": fbm_test.std(ddof=1),
    "fgn_lag1_autocorr": pd.Series(fgn_test).autocorr(1),
})

## 6. Simulate rough and smooth log-volatility paths

We create two synthetic volatility processes:

### Rough log-volatility

$$
\begin{aligned}
\log \sigma_t &= \log \bar \sigma \\
&\quad + \nu W_t^H
\end{aligned}
$$

with $H=0.12$.

### Smooth/Brownian-like benchmark

Same structure but with $H=0.5$.

Both are rescaled to have comparable average volatility.

In [ ]:
def simulate_logvol_path(config, hurst, label):
    rng = np.random.default_rng(config.seed + int(hurst * 1000) + len(label))
    fbm, fgn = simulate_fractional_brownian_motion(config.n_days, hurst, rng)

    log_base_daily_vol = np.log(config.base_ann_vol / np.sqrt(config.annualisation))
    logvol = log_base_daily_vol + config.vol_of_logvol * fbm

    # Recentre so average annualised vol is near target.
    daily_vol = np.exp(logvol)
    scaling = (config.base_ann_vol / np.sqrt(config.annualisation)) / daily_vol.mean()
    daily_vol = daily_vol * scaling
    logvol = np.log(daily_vol)

    dates = pd.bdate_range("2020-01-01", periods=config.n_days)

    out = pd.DataFrame({
        "date": dates,
        "log_vol": logvol,
        "daily_vol": daily_vol,
        "annualised_vol": daily_vol * np.sqrt(config.annualisation),
        "fbm": fbm,
        "fgn": fgn,
        "hurst_true": hurst,
        "process": label,
    }).set_index("date")

    return out

rough_vol = simulate_logvol_path(config, config.true_rough_hurst, "rough")
smooth_vol = simulate_logvol_path(config, config.smooth_hurst_proxy, "smooth")

rough_vol.head(), smooth_vol.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(rough_vol.index, rough_vol["annualised_vol"], label="rough vol")
plt.plot(smooth_vol.index, smooth_vol["annualised_vol"], label="smooth benchmark", alpha=0.75)
plt.title("Simulated Annualised Volatility")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(rough_vol.index, rough_vol["log_vol"], label="rough log-vol")
plt.plot(smooth_vol.index, smooth_vol["log_vol"], label="smooth log-vol", alpha=0.75)
plt.title("Simulated Log-Volatility")
plt.xlabel("Date")
plt.ylabel("log daily vol")
plt.legend()
plt.show()

## 7. Simulate intraday returns

Realised variance is built from intraday returns:

$$
RV_t = \sum_{i=1}^{M} r_{t,i}^2
$$

We simulate intraday returns conditional on daily volatility.

For each day:

$$
\begin{aligned}
r_{t,i} &= \frac{\mu}{M} \\
&\quad + \frac{\sigma_t}{\sqrt{M}} \epsilon_{t,i}
\end{aligned}
$$

Then add optional microstructure noise to observed prices.

In [ ]:
def simulate_intraday_returns(vol_df, config, add_microstructure_noise=False):
    rng = np.random.default_rng(config.seed + (999 if add_microstructure_noise else 111))
    dates = vol_df.index
    rows = []

    price = 100.0
    for date in dates:
        daily_vol = vol_df.loc[date, "daily_vol"]
        intraday_sigma = daily_vol / np.sqrt(config.intraday_steps)
        intraday_mu = config.drift_ann / config.annualisation / config.intraday_steps

        latent_returns = intraday_mu + intraday_sigma * rng.standard_normal(config.intraday_steps)

        latent_log_prices = np.log(price) + np.cumsum(latent_returns)
        latent_prices = np.exp(latent_log_prices)

        if add_microstructure_noise:
            noise = rng.normal(0.0, config.microstructure_noise_bps / 10000.0, size=config.intraday_steps)
            observed_prices = latent_prices * np.exp(noise)
            observed_returns = np.diff(np.r_[price, observed_prices]) / np.r_[price, observed_prices[:-1]]
        else:
            observed_returns = latent_returns
            observed_prices = latent_prices

        for i in range(config.intraday_steps):
            rows.append({
                "date": date,
                "bar": i,
                "latent_return": latent_returns[i],
                "observed_return": observed_returns[i],
                "latent_price": latent_prices[i],
                "observed_price": observed_prices[i] if add_microstructure_noise else latent_prices[i],
                "process": vol_df["process"].iloc[0],
                "microstructure_noise": add_microstructure_noise,
            })

        price = latent_prices[-1]

    return pd.DataFrame(rows)

rough_intraday_clean = simulate_intraday_returns(rough_vol, config, add_microstructure_noise=False)
rough_intraday_noisy = simulate_intraday_returns(rough_vol, config, add_microstructure_noise=True)
smooth_intraday_clean = simulate_intraday_returns(smooth_vol, config, add_microstructure_noise=False)

rough_intraday_clean.head()

## 8. Realised variance and realised volatility

Compute:

$$
RV_t = \sum_i r_{t,i}^2
$$

$$
RVol_t = \sqrt{RV_t}
$$

Then use:

$$
X_t = \log(RV_t)
$$

as the empirical log-volatility proxy.

In [ ]:
def realised_volatility_from_intraday(intraday_df, return_col="observed_return"):
    rv = (
        intraday_df
        .groupby("date")
        .agg(
            realised_variance=(return_col, lambda x: np.sum(np.asarray(x) ** 2)),
            realised_return=(return_col, "sum"),
            n_bars=(return_col, "count"),
        )
    )

    rv["realised_vol"] = np.sqrt(rv["realised_variance"])
    rv["log_rv"] = np.log(rv["realised_variance"].clip(lower=1e-16))
    rv["ann_realised_vol"] = rv["realised_vol"] * np.sqrt(config.annualisation)
    rv["process"] = intraday_df["process"].iloc[0]
    rv["microstructure_noise"] = intraday_df["microstructure_noise"].iloc[0]
    return rv

rv_rough_clean = realised_volatility_from_intraday(rough_intraday_clean)
rv_rough_noisy = realised_volatility_from_intraday(rough_intraday_noisy)
rv_smooth_clean = realised_volatility_from_intraday(smooth_intraday_clean)

rv_rough_clean.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(rv_rough_clean.index, rv_rough_clean["ann_realised_vol"], label="rough clean RV")
plt.plot(rv_rough_noisy.index, rv_rough_noisy["ann_realised_vol"], label="rough noisy RV", alpha=0.75)
plt.plot(rv_smooth_clean.index, rv_smooth_clean["ann_realised_vol"], label="smooth clean RV", alpha=0.75)
plt.title("Realised Volatility Estimates")
plt.xlabel("Date")
plt.ylabel("Annualised realised vol")
plt.legend()
plt.show()

## 9. Empirical variogram

For each lag $\ell$:

$$
m(q,\ell) = \frac{1}{T-\ell} \sum_t |X_{t+\ell}-X_t|^q
$$

We estimate for $q=1$ and $q=2$.

For rough volatility:

$$
\log m(q,\ell) \approx c + qH\log \ell
$$

In [ ]:
def empirical_variogram(series, max_lag, qs=(1, 2)):
    x = pd.Series(series).dropna().to_numpy()
    rows = []

    for lag in range(1, max_lag + 1):
        diff = x[lag:] - x[:-lag]
        for q in qs:
            moment = np.mean(np.abs(diff) ** q)
            rows.append({
                "lag": lag,
                "q": q,
                "moment": moment,
                "log_lag": np.log(lag),
                "log_moment": np.log(max(moment, 1e-18)),
            })

    return pd.DataFrame(rows)

variogram_rough_clean = empirical_variogram(rv_rough_clean["log_rv"], config.max_variogram_lag)
variogram_rough_noisy = empirical_variogram(rv_rough_noisy["log_rv"], config.max_variogram_lag)
variogram_smooth_clean = empirical_variogram(rv_smooth_clean["log_rv"], config.max_variogram_lag)

variogram_rough_clean.head()

In [ ]:
plt.figure(figsize=(10, 6))
for label, vg in [
    ("rough clean", variogram_rough_clean),
    ("rough noisy", variogram_rough_noisy),
    ("smooth clean", variogram_smooth_clean),
]:
    sub = vg[vg["q"] == 2]
    plt.plot(sub["log_lag"], sub["log_moment"], marker="o", label=label)
plt.title("Log-Log Variogram, q=2")
plt.xlabel("log lag")
plt.ylabel("log moment")
plt.legend()
plt.show()

## 10. Hurst estimation from variogram slope

For each $q$:

$$
\hat H_q = \frac{\hat{slope}_q}{q}
$$

where:

$$
\log m(q,\ell) = c_q + \hat{slope}_q \log \ell
$$

We estimate over a chosen lag range. Small lags can be noisy; large lags can be contaminated by finite-sample and nonstationarity effects.

In [ ]:
def estimate_hurst_from_variogram(variogram, min_lag=1, max_lag=30):
    rows = []

    for q, group in variogram.groupby("q"):
        sub = group[(group["lag"] >= min_lag) & (group["lag"] <= max_lag)].copy()
        X = np.column_stack([np.ones(len(sub)), sub["log_lag"].to_numpy()])
        y = sub["log_moment"].to_numpy()

        beta = np.linalg.pinv(X.T @ X) @ X.T @ y
        fitted = X @ beta
        residual = y - fitted

        ss_res = residual @ residual
        ss_tot = (y - y.mean()) @ (y - y.mean())
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

        slope = beta[1]
        h_hat = slope / q

        rows.append({
            "q": q,
            "min_lag": min_lag,
            "max_lag": max_lag,
            "slope": slope,
            "hurst_estimate": h_hat,
            "intercept": beta[0],
            "r2": r2,
            "n_lags": len(sub),
        })

    return pd.DataFrame(rows)

hurst_rough_clean = estimate_hurst_from_variogram(variogram_rough_clean, 1, 30)
hurst_rough_noisy = estimate_hurst_from_variogram(variogram_rough_noisy, 1, 30)
hurst_smooth_clean = estimate_hurst_from_variogram(variogram_smooth_clean, 1, 30)

hurst_estimates = pd.concat([
    hurst_rough_clean.assign(series="rough_clean", true_hurst=config.true_rough_hurst),
    hurst_rough_noisy.assign(series="rough_noisy", true_hurst=config.true_rough_hurst),
    hurst_smooth_clean.assign(series="smooth_clean", true_hurst=config.smooth_hurst_proxy),
], ignore_index=True)

hurst_estimates

## 11. Lag-range sensitivity

Hurst estimates can change depending on which lags are used.

We test multiple lag windows.

In [ ]:
def lag_range_sensitivity(variogram, label, true_hurst):
    rows = []
    lag_windows = [(1, 10), (1, 20), (1, 30), (2, 30), (5, 40), (10, 60)]

    for min_lag, max_lag in lag_windows:
        est = estimate_hurst_from_variogram(variogram, min_lag, max_lag)
        est["series"] = label
        est["true_hurst"] = true_hurst
        rows.append(est)

    return pd.concat(rows, ignore_index=True)

lag_sensitivity = pd.concat([
    lag_range_sensitivity(variogram_rough_clean, "rough_clean", config.true_rough_hurst),
    lag_range_sensitivity(variogram_rough_noisy, "rough_noisy", config.true_rough_hurst),
    lag_range_sensitivity(variogram_smooth_clean, "smooth_clean", config.smooth_hurst_proxy),
], ignore_index=True)

lag_sensitivity

In [ ]:
plt.figure(figsize=(12, 6))
for series, sub in lag_sensitivity[lag_sensitivity["q"] == 2].groupby("series"):
    labels = sub["min_lag"].astype(str) + "-" + sub["max_lag"].astype(str)
    plt.plot(labels, sub["hurst_estimate"], marker="o", label=series)
plt.axhline(config.true_rough_hurst, linestyle="--", label="true rough H")
plt.axhline(config.smooth_hurst_proxy, linestyle=":", label="smooth H")
plt.title("Hurst Estimate Sensitivity by Lag Window, q=2")
plt.xlabel("Lag window")
plt.ylabel("Estimated H")
plt.xticks(rotation=45)
plt.legend()
plt.show()

## 12. Bootstrap confidence intervals

We use a block bootstrap over log-RV observations.

This preserves local dependence better than iid resampling.

In [ ]:
def moving_block_bootstrap_series(series, block_length, rng):
    x = pd.Series(series).dropna().to_numpy()
    n = len(x)
    out = []

    while len(out) < n:
        start = rng.integers(0, n)
        block = [x[(start + j) % n] for j in range(block_length)]
        out.extend(block)

    return pd.Series(out[:n])

def bootstrap_hurst(series, config, min_lag=1, max_lag=30, q=2, block_length=20):
    rng = np.random.default_rng(config.seed + 2025)
    rows = []

    for b in range(config.bootstrap_samples):
        boot_series = moving_block_bootstrap_series(series, block_length, rng)
        vg = empirical_variogram(boot_series, config.max_variogram_lag, qs=(q,))
        est = estimate_hurst_from_variogram(vg, min_lag, max_lag)
        rows.append({
            "bootstrap_id": b,
            "hurst_estimate": est["hurst_estimate"].iloc[0],
            "slope": est["slope"].iloc[0],
            "r2": est["r2"].iloc[0],
        })

    return pd.DataFrame(rows)

bootstrap_rough = bootstrap_hurst(rv_rough_clean["log_rv"], config, min_lag=1, max_lag=30, q=2)
bootstrap_smooth = bootstrap_hurst(rv_smooth_clean["log_rv"], config, min_lag=1, max_lag=30, q=2)

bootstrap_ci = pd.DataFrame([
    {
        "series": "rough_clean",
        "H_mean": bootstrap_rough["hurst_estimate"].mean(),
        "H_p05": bootstrap_rough["hurst_estimate"].quantile(0.05),
        "H_p50": bootstrap_rough["hurst_estimate"].quantile(0.50),
        "H_p95": bootstrap_rough["hurst_estimate"].quantile(0.95),
    },
    {
        "series": "smooth_clean",
        "H_mean": bootstrap_smooth["hurst_estimate"].mean(),
        "H_p05": bootstrap_smooth["hurst_estimate"].quantile(0.05),
        "H_p50": bootstrap_smooth["hurst_estimate"].quantile(0.50),
        "H_p95": bootstrap_smooth["hurst_estimate"].quantile(0.95),
    },
])

bootstrap_ci

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(bootstrap_rough["hurst_estimate"], bins=40, alpha=0.6, label="rough")
plt.hist(bootstrap_smooth["hurst_estimate"], bins=40, alpha=0.6, label="smooth")
plt.axvline(config.true_rough_hurst, linestyle="--", label="true rough H")
plt.axvline(config.smooth_hurst_proxy, linestyle=":", label="smooth H")
plt.title("Bootstrap Hurst Estimates")
plt.xlabel("Estimated H")
plt.ylabel("Count")
plt.legend()
plt.show()

## 13. Microstructure noise bias

Microstructure noise can distort realised variance.

At very high sampling frequency, observed price noise can inflate realised variance.

We compare clean and noisy realised volatility estimates and Hurst estimates.

In [ ]:
microstructure_bias_report = pd.DataFrame([{
    "series": "rough_clean",
    "mean_ann_rv": rv_rough_clean["ann_realised_vol"].mean(),
    "std_log_rv": rv_rough_clean["log_rv"].std(),
    "hurst_q2_1_30": hurst_rough_clean[hurst_rough_clean["q"] == 2]["hurst_estimate"].iloc[0],
},
{
    "series": "rough_noisy",
    "mean_ann_rv": rv_rough_noisy["ann_realised_vol"].mean(),
    "std_log_rv": rv_rough_noisy["log_rv"].std(),
    "hurst_q2_1_30": hurst_rough_noisy[hurst_rough_noisy["q"] == 2]["hurst_estimate"].iloc[0],
}])

microstructure_bias_report["mean_ann_rv_ratio_vs_clean"] = (
    microstructure_bias_report["mean_ann_rv"]
    / microstructure_bias_report.loc[microstructure_bias_report["series"] == "rough_clean", "mean_ann_rv"].iloc[0]
)

microstructure_bias_report

## 14. Subsampled realised volatility

A simple noise mitigation method is to sample less frequently.

This reduces microstructure noise but loses information.

We compute realised variance using every $k$-th intraday bar.

In [ ]:
def subsampled_realised_vol(intraday_df, step):
    rows = []

    for date, group in intraday_df.groupby("date"):
        group = group.sort_values("bar")
        prices = group["observed_price"].to_numpy()
        sampled = prices[::step]
        if len(sampled) < 2:
            continue
        returns = np.diff(np.log(sampled))
        rv = np.sum(returns**2)
        rows.append({
            "date": date,
            "subsample_step": step,
            "realised_variance": rv,
            "realised_vol": np.sqrt(rv),
            "ann_realised_vol": np.sqrt(rv) * np.sqrt(config.annualisation),
            "log_rv": np.log(max(rv, 1e-16)),
        })

    return pd.DataFrame(rows).set_index("date")

subsample_rows = []
for step in [1, 2, 3, 6, 13]:
    rv_sub = subsampled_realised_vol(rough_intraday_noisy, step)
    vg = empirical_variogram(rv_sub["log_rv"], config.max_variogram_lag)
    h = estimate_hurst_from_variogram(vg, 1, 30)
    subsample_rows.append({
        "subsample_step": step,
        "bars_used_per_day_approx": config.intraday_steps // step,
        "mean_ann_rv": rv_sub["ann_realised_vol"].mean(),
        "hurst_q2": h[h["q"] == 2]["hurst_estimate"].iloc[0],
    })

subsample_noise_report = pd.DataFrame(subsample_rows)

subsample_noise_report

## 15. Roughness diagnostics table

We combine diagnostics:

- estimated H;
- confidence interval;
- lag sensitivity;
- microstructure sensitivity;
- roughness classification.

In [ ]:
def classify_roughness(h):
    if pd.isna(h):
        return "unknown"
    if h < 0.25:
        return "rough"
    if h < 0.40:
        return "moderately_rough"
    if h < 0.60:
        return "brownian_like"
    return "smooth_persistent"

roughness_diagnostics = hurst_estimates[hurst_estimates["q"] == 2].copy()
roughness_diagnostics = roughness_diagnostics.merge(bootstrap_ci, on="series", how="left")
roughness_diagnostics["roughness_classification"] = roughness_diagnostics["hurst_estimate"].map(classify_roughness)
roughness_diagnostics["absolute_error_vs_true"] = (roughness_diagnostics["hurst_estimate"] - roughness_diagnostics["true_hurst"]).abs()

roughness_diagnostics

## 16. Volatility forecasting: HAR-RV baseline

The HAR-RV model uses daily, weekly, and monthly realised variance components:

$$
\begin{aligned}
RV_{t+1} &= \beta_0 \\
&\quad + \beta_d RV_t \\
&\quad + \beta_w \overline{RV}_{t-4:t} \\
&\quad + \beta_m \overline{RV}_{t-21:t} \\
&\quad + \epsilon_{t+1}
\end{aligned}
$$

We compare this to a rough-volatility-inspired forecast using power-law weights.

In [ ]:
def build_forecast_dataset(rv_df):
    df = rv_df.copy()
    df["target_log_rv_next"] = df["log_rv"].shift(-1)
    df["log_rv_d"] = df["log_rv"]
    df["log_rv_w"] = df["log_rv"].rolling(5).mean()
    df["log_rv_m"] = df["log_rv"].rolling(22).mean()
    return df.dropna()

def ols_predict(train_X, train_y, test_X):
    X = np.column_stack([np.ones(len(train_X)), train_X])
    beta = np.linalg.pinv(X.T @ X) @ X.T @ train_y
    pred = np.column_stack([np.ones(len(test_X)), test_X]) @ beta
    return pred, beta

def power_law_weights(n_lags, hurst):
    lags = np.arange(1, n_lags + 1)
    weights = lags ** (hurst - 0.5)
    weights = weights / weights.sum()
    return weights

def rough_powerlaw_forecast_features(log_rv, n_lags, hurst):
    weights = power_law_weights(n_lags, hurst)
    values = []
    idx = []

    for i in range(n_lags, len(log_rv)):
        past = log_rv.iloc[i - n_lags:i].to_numpy()[::-1]
        values.append(np.sum(weights * past))
        idx.append(log_rv.index[i])

    return pd.Series(values, index=idx, name="rough_powerlaw_feature")

def forecast_comparison(rv_df, estimated_hurst):
    df = build_forecast_dataset(rv_df)

    split = int(len(df) * config.forecast_train_frac)
    train = df.iloc[:split]
    test = df.iloc[split:]

    har_cols = ["log_rv_d", "log_rv_w", "log_rv_m"]
    har_pred, har_beta = ols_predict(train[har_cols].to_numpy(), train["target_log_rv_next"].to_numpy(), test[har_cols].to_numpy())

    power_feature = rough_powerlaw_forecast_features(rv_df["log_rv"], n_lags=63, hurst=max(estimated_hurst, 0.01))
    df_power = df.join(power_feature, how="inner").dropna()
    split_power = int(len(df_power) * config.forecast_train_frac)
    train_p = df_power.iloc[:split_power]
    test_p = df_power.iloc[split_power:]

    rough_pred, rough_beta = ols_predict(
        train_p[["rough_powerlaw_feature"]].to_numpy(),
        train_p["target_log_rv_next"].to_numpy(),
        test_p[["rough_powerlaw_feature"]].to_numpy(),
    )

    har_eval = pd.DataFrame({
        "date": test.index,
        "model": "HAR_log_RV",
        "actual": test["target_log_rv_next"].to_numpy(),
        "prediction": har_pred,
    })

    rough_eval = pd.DataFrame({
        "date": test_p.index,
        "model": "rough_powerlaw_log_RV",
        "actual": test_p["target_log_rv_next"].to_numpy(),
        "prediction": rough_pred,
    })

    eval_df = pd.concat([har_eval, rough_eval], ignore_index=True)
    eval_df["error"] = eval_df["actual"] - eval_df["prediction"]
    eval_df["squared_error"] = eval_df["error"] ** 2
    eval_df["absolute_error"] = eval_df["error"].abs()

    summary = (
        eval_df
        .groupby("model")
        .agg(
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            mae=("absolute_error", "mean"),
            bias=("error", "mean"),
            n_test=("error", "count"),
        )
        .reset_index()
    )

    coef = {
        "har_beta": har_beta.tolist(),
        "rough_beta": rough_beta.tolist(),
    }

    return eval_df, summary, coef

estimated_rough_h = hurst_rough_clean[hurst_rough_clean["q"] == 2]["hurst_estimate"].iloc[0]
forecast_eval, forecast_summary, forecast_coefficients = forecast_comparison(rv_rough_clean, estimated_rough_h)

forecast_summary

In [ ]:
plt.figure(figsize=(12, 5))
for model, group in forecast_eval.groupby("model"):
    ordered = group.sort_values("date")
    plt.plot(ordered["date"], ordered["prediction"], label=f"{model} forecast", alpha=0.8)
plt.plot(
    forecast_eval.sort_values("date").drop_duplicates("date")["date"],
    forecast_eval.sort_values("date").drop_duplicates("date")["actual"],
    label="actual",
    color="black",
    alpha=0.45,
)
plt.title("Log-RV Forecast Comparison")
plt.xlabel("Date")
plt.ylabel("Next-day log RV")
plt.legend()
plt.show()

## 17. Execution-risk implications

Execution cost models often scale with volatility:

$$
Impact \propto \sigma \sqrt{Q/V}
$$

If volatility changes roughly and quickly, execution cost estimates can become stale.

We compare:

- realised volatility;
- one-day forecast error;
- impact estimate error for a fixed participation trade.

In [ ]:
def execution_impact_error_from_vol_forecast(rv_df, forecast_eval, participation=0.10, y=0.75):
    har = forecast_eval[forecast_eval["model"] == "HAR_log_RV"].copy()
    har["predicted_rv"] = np.exp(har["prediction"])
    har["actual_rv"] = np.exp(har["actual"])
    har["predicted_vol"] = np.sqrt(har["predicted_rv"])
    har["actual_vol"] = np.sqrt(har["actual_rv"])

    har["predicted_impact_bps"] = y * har["predicted_vol"] * np.sqrt(participation) * 10000
    har["actual_impact_bps"] = y * har["actual_vol"] * np.sqrt(participation) * 10000
    har["impact_error_bps"] = har["actual_impact_bps"] - har["predicted_impact_bps"]

    summary = pd.DataFrame([{
        "participation": participation,
        "mean_predicted_impact_bps": har["predicted_impact_bps"].mean(),
        "mean_actual_impact_bps": har["actual_impact_bps"].mean(),
        "mean_impact_error_bps": har["impact_error_bps"].mean(),
        "mae_impact_error_bps": har["impact_error_bps"].abs().mean(),
        "p95_abs_impact_error_bps": har["impact_error_bps"].abs().quantile(0.95),
    }])

    return har, summary

impact_error, impact_error_summary = execution_impact_error_from_vol_forecast(rv_rough_clean, forecast_eval, participation=0.10, y=0.75)

impact_error_summary

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(impact_error["date"], impact_error["impact_error_bps"])
plt.axhline(0, linestyle="--")
plt.title("Execution Impact Error from Vol Forecast Error")
plt.xlabel("Date")
plt.ylabel("Actual impact - predicted impact, bps")
plt.show()

## 18. Volatility-of-volatility and roughness

Rough volatility is associated with large short-horizon changes in volatility.

We compute:

- daily absolute log-RV changes;
- volatility-of-volatility proxy;
- jump-like log-RV moves.

In [ ]:
def vol_of_vol_diagnostics(rv_df, label):
    log_rv = rv_df["log_rv"].dropna()
    diff = log_rv.diff().dropna()

    threshold = diff.abs().quantile(0.99)

    return pd.DataFrame([{
        "series": label,
        "std_log_rv_change": diff.std(ddof=1),
        "mean_abs_log_rv_change": diff.abs().mean(),
        "p95_abs_log_rv_change": diff.abs().quantile(0.95),
        "p99_abs_log_rv_change": threshold,
        "n_jump_like_moves_99pct": int((diff.abs() >= threshold).sum()),
        "lag1_log_rv_change_autocorr": diff.autocorr(1),
    }])

vov_diagnostics = pd.concat([
    vol_of_vol_diagnostics(rv_rough_clean, "rough_clean"),
    vol_of_vol_diagnostics(rv_smooth_clean, "smooth_clean"),
    vol_of_vol_diagnostics(rv_rough_noisy, "rough_noisy"),
], ignore_index=True)

vov_diagnostics

## 19. Governance flags

A rough volatility estimator should be reviewed if:

1. Hurst estimate is highly sensitive to lag window;
2. bootstrap interval is too wide;
3. microstructure noise materially changes $H$;
4. variogram fit quality is poor;
5. volatility forecast error creates large impact error;
6. estimator is applied to too little data.

In [ ]:
rough_q2_lag = lag_sensitivity[(lag_sensitivity["series"] == "rough_clean") & (lag_sensitivity["q"] == 2)]
hurst_range = rough_q2_lag["hurst_estimate"].max() - rough_q2_lag["hurst_estimate"].min()

rough_ci_row = bootstrap_ci[bootstrap_ci["series"] == "rough_clean"].iloc[0]
ci_width = rough_ci_row["H_p95"] - rough_ci_row["H_p05"]

h_clean = hurst_rough_clean[hurst_rough_clean["q"] == 2]["hurst_estimate"].iloc[0]
h_noisy = hurst_rough_noisy[hurst_rough_noisy["q"] == 2]["hurst_estimate"].iloc[0]
noise_shift = abs(h_noisy - h_clean)

fit_r2 = hurst_rough_clean[hurst_rough_clean["q"] == 2]["r2"].iloc[0]
p95_impact_error = impact_error_summary["p95_abs_impact_error_bps"].iloc[0]

governance_flags = pd.DataFrame([{
    "estimated_H_q2_clean": h_clean,
    "estimated_H_q2_noisy": h_noisy,
    "hurst_lag_range_sensitivity": hurst_range,
    "bootstrap_ci_width": ci_width,
    "microstructure_noise_H_shift": noise_shift,
    "variogram_fit_r2": fit_r2,
    "p95_abs_impact_error_bps": p95_impact_error,
    "n_days": config.n_days,
    "flag_lag_sensitivity_high": bool(hurst_range > 0.20),
    "flag_bootstrap_ci_wide": bool(ci_width > 0.25),
    "flag_microstructure_noise_shift_high": bool(noise_shift > 0.15),
    "flag_variogram_fit_poor": bool(fit_r2 < 0.85),
    "flag_impact_error_high": bool(p95_impact_error > 25.0),
    "flag_short_sample": bool(config.n_days < 500),
    "flag_synthetic_data_not_real_market": True,
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_lag_sensitivity_high",
        "flag_bootstrap_ci_wide",
        "flag_microstructure_noise_shift_high",
        "flag_variogram_fit_poor",
        "flag_impact_error_high",
        "flag_short_sample",
        "flag_synthetic_data_not_real_market",
    ]
].any(axis=1)

governance_flags

## 20. Audit checks

Numerical and process checks:

1. realised variance is positive;
2. variogram moments are positive;
3. Hurst estimates are finite;
4. bootstrap estimates are finite;
5. forecast outputs are finite;
6. impact error outputs are finite;
7. governance flags exist.

In [ ]:
audit_rows = []

rv_positive = bool(
    (rv_rough_clean["realised_variance"] > 0).all()
    and (rv_smooth_clean["realised_variance"] > 0).all()
)
audit_rows.append({
    "check": "realised_variance_positive",
    "value": rv_positive,
    "passed": rv_positive,
})

variogram_positive = bool(
    (variogram_rough_clean["moment"] > 0).all()
    and (variogram_smooth_clean["moment"] > 0).all()
)
audit_rows.append({
    "check": "variogram_moments_positive",
    "value": variogram_positive,
    "passed": variogram_positive,
})

hurst_finite = bool(np.isfinite(hurst_estimates[["slope", "hurst_estimate", "r2"]].to_numpy()).all())
audit_rows.append({
    "check": "hurst_estimates_finite",
    "value": hurst_finite,
    "passed": hurst_finite,
})

bootstrap_finite = bool(
    np.isfinite(bootstrap_rough[["hurst_estimate", "slope", "r2"]].to_numpy()).all()
    and np.isfinite(bootstrap_smooth[["hurst_estimate", "slope", "r2"]].to_numpy()).all()
)
audit_rows.append({
    "check": "bootstrap_outputs_finite",
    "value": bootstrap_finite,
    "passed": bootstrap_finite,
})

forecast_finite = bool(np.isfinite(forecast_eval[["actual", "prediction", "error"]].to_numpy()).all())
audit_rows.append({
    "check": "forecast_outputs_finite",
    "value": forecast_finite,
    "passed": forecast_finite,
})

impact_error_finite = bool(np.isfinite(impact_error_summary.select_dtypes(include=[float, int]).to_numpy()).all())
audit_rows.append({
    "check": "impact_error_summary_finite",
    "value": impact_error_finite,
    "passed": impact_error_finite,
})

governance_exists = bool(len(governance_flags) == 1)
audit_rows.append({
    "check": "governance_flags_exist",
    "value": governance_exists,
    "passed": governance_exists,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 21. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed") / config.output_subdir
output_dir.mkdir(parents=True, exist_ok=True)

rough_vol.to_csv(output_dir / "rough_vol_path.csv")
smooth_vol.to_csv(output_dir / "smooth_vol_path.csv")
rough_intraday_clean.to_csv(output_dir / "rough_intraday_clean.csv", index=False)
rough_intraday_noisy.to_csv(output_dir / "rough_intraday_noisy.csv", index=False)
smooth_intraday_clean.to_csv(output_dir / "smooth_intraday_clean.csv", index=False)

rv_rough_clean.to_csv(output_dir / "rv_rough_clean.csv")
rv_rough_noisy.to_csv(output_dir / "rv_rough_noisy.csv")
rv_smooth_clean.to_csv(output_dir / "rv_smooth_clean.csv")

variogram_rough_clean.to_csv(output_dir / "variogram_rough_clean.csv", index=False)
variogram_rough_noisy.to_csv(output_dir / "variogram_rough_noisy.csv", index=False)
variogram_smooth_clean.to_csv(output_dir / "variogram_smooth_clean.csv", index=False)

hurst_estimates.to_csv(output_dir / "hurst_estimates.csv", index=False)
lag_sensitivity.to_csv(output_dir / "lag_range_sensitivity.csv", index=False)
bootstrap_rough.to_csv(output_dir / "bootstrap_hurst_rough.csv", index=False)
bootstrap_smooth.to_csv(output_dir / "bootstrap_hurst_smooth.csv", index=False)
bootstrap_ci.to_csv(output_dir / "bootstrap_hurst_ci.csv", index=False)
microstructure_bias_report.to_csv(output_dir / "microstructure_bias_report.csv", index=False)
subsample_noise_report.to_csv(output_dir / "subsample_noise_report.csv", index=False)
roughness_diagnostics.to_csv(output_dir / "roughness_diagnostics.csv", index=False)
forecast_eval.to_csv(output_dir / "forecast_eval.csv", index=False)
forecast_summary.to_csv(output_dir / "forecast_summary.csv", index=False)
pd.Series(forecast_coefficients).to_frame("value").to_csv(output_dir / "forecast_coefficients.csv")
impact_error.to_csv(output_dir / "execution_impact_error.csv", index=False)
impact_error_summary.to_csv(output_dir / "execution_impact_error_summary.csv", index=False)
vov_diagnostics.to_csv(output_dir / "vol_of_vol_diagnostics.csv", index=False)
governance_flags.to_csv(output_dir / "governance_flags.csv", index=False)
audit_report.to_csv(output_dir / "audit_report.csv", index=False)

manifest = {
    "dataset_name": "rough_volatility_estimation_outputs",
    "schema_version": "rough_volatility_estimation_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "hurst_estimates": hurst_estimates.to_dict(orient="records"),
    "bootstrap_ci": bootstrap_ci.to_dict(orient="records"),
    "roughness_diagnostics": roughness_diagnostics.to_dict(orient="records"),
    "forecast_summary": forecast_summary.to_dict(orient="records"),
    "impact_error_summary": impact_error_summary.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "This notebook estimates roughness from log-realised-volatility variograms.",
        "Synthetic rough and smooth volatility paths are compared.",
        "Hurst estimation is tested for lag-window sensitivity and bootstrap uncertainty.",
        "Microstructure noise and subsampling effects are demonstrated.",
        "HAR-RV and rough-power-law volatility forecasts are compared.",
        "Execution impact error from volatility forecast error is estimated.",
        "The data is synthetic and should be replaced with real high-frequency realised-volatility data for production research."
    ],
}

manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

output_dir / "hurst_estimates.csv", output_dir / "forecast_summary.csv", output_dir / "governance_flags.csv", manifest_path

## 22. Practical checklist for rough volatility estimation

Before trusting a rough volatility estimate:

1. **Use a clear volatility proxy.**
2. **Report sampling frequency.**
3. **Check microstructure noise sensitivity.**
4. **Estimate H over multiple lag windows.**
5. **Bootstrap confidence intervals.**
6. **Compare rough and smooth benchmarks.**
7. **Check variogram fit quality.**
8. **Test forecast implications out of sample.**
9. **Connect vol forecast errors to risk and execution costs.**
10. **Avoid over-interpreting a single H estimate.**

## 23. Limitations

### Synthetic data

The notebook uses simulated volatility and returns. Real high-frequency data has market microstructure noise, jumps, trading halts, calendar effects and missing data.

### Cholesky simulation

Fractional Gaussian noise is simulated with Cholesky decomposition. This is transparent but not the fastest method.

### Simple realised volatility

Realised variance is estimated from evenly spaced intraday bars without jump-robust estimators.

### Hurst estimation bias

Variogram-based H estimates are sensitive to lag range, sample size, noise and nonstationarity.

### No option calibration

The notebook does not calibrate rough Bergomi or rough Heston to option surfaces.

### Forecast comparison is simple

The rough-power-law forecast is illustrative, not a full rough-volatility forecasting model.

## 24. Research frontier and extensions

Important extensions include:

1. rough Bergomi simulation;
2. rough Heston calibration;
3. hybrid schemes for fractional kernels;
4. jump-robust realised volatility;
5. pre-averaging and realised kernels;
6. option-implied roughness;
7. multifractal volatility;
8. intraday roughness by session;
9. rough volatility in futures markets;
10. execution algorithms that adapt to rough volatility forecasts.

## 25. Suggested follow-up notebooks

This notebook naturally leads to:

1. **06_06_execution_algorithms_twap_vwap_pov**  
   Use volatility forecasts in execution-rate decisions.

2. **06_07_latency_and_queue_position_model**  
   Connect rough volatility to short-horizon fill risk.

3. **06_08_microprice_and_short_horizon_alpha**  
   Combine rough volatility with microprice signals.

4. **06_09_execution_risk_controls_and_kill_switch**  
   Trigger controls when volatility forecasts jump.

5. **07_04_rough_volatility_options_capstone**  
   Extend roughness estimation to option-pricing models.

## 26. Summary

This notebook implemented rough volatility estimation.

It showed how to:

1. simulate fractional Gaussian noise;
2. simulate rough and smooth log-volatility paths;
3. simulate intraday returns;
4. compute realised variance and realised volatility;
5. build log-RV variograms;
6. estimate Hurst exponents;
7. test lag-range sensitivity;
8. bootstrap confidence intervals;
9. demonstrate microstructure noise bias;
10. test subsampling effects;
11. classify roughness;
12. compare HAR-RV with rough-power-law forecasts;
13. estimate execution impact error from volatility forecast error;
14. compute volatility-of-volatility diagnostics;
15. create governance flags and audit checks;
16. save outputs and manifests.

The key computational takeaway:

> Roughness is estimated from how log-volatility increments scale across time lags.

The key financial takeaway:

> If volatility is rough, risk and execution-cost estimates can become stale quickly, especially at short horizons.

## 27. Further reading

- Gatheral, Jaisson and Rosenbaum, "Volatility is Rough."
- Bennedsen, Lunde and Pakkanen on hybrid simulation schemes for Brownian semistationary processes.
- Bayer, Friz and Gatheral on rough volatility.
- El Euch, Fukasawa and Rosenbaum on rough Heston.
- Andersen et al. on realised volatility.
- Barndorff-Nielsen and Shephard on realised variation.